# 00 — Ingest

Pull raw StatsBomb open data for **Premier League 2015/16**
(`competition_id=2`, `season_id=27`), 380 matches. Writes:

- `data/statsbomb.csv` — raw `statsbombpy` events, one row per event, with a `match_id` column
- `data/event_df.csv` — SPADL actions (`socceraction`), with action-type / result / team / player names

Hits the network; takes a while.

In [ ]:
import warnings
from statsbombpy.api_client import NoAuthWarning

warnings.filterwarnings("ignore", category=NoAuthWarning)
warnings.filterwarnings("ignore", message="credentials were not supplied")

import pandas as pd
from statsbombpy import sb
from socceraction.data.statsbomb import StatsBombLoader
import socceraction.spadl as spadl

COMPETITION_ID = 2
SEASON_ID = 27

api = StatsBombLoader(getter="remote", creds=None)

fixtures = api.games(competition_id=COMPETITION_ID, season_id=SEASON_ID)
match_ids = list(fixtures["game_id"].unique())
print(f"{len(match_ids)} matches")

## Raw `statsbombpy` events

In [ ]:
raw = []
for i, match_id in enumerate(match_ids, 1):
    events = sb.events(match_id=match_id)
    events["match_id"] = match_id
    raw.append(events)
    print(f"  raw events {i}/{len(match_ids)}  match {match_id}  ({len(events)} rows)")

df = pd.concat(raw, ignore_index=True)
df.to_csv("data/statsbomb.csv")
print(f"data/statsbomb.csv  {df.shape}")

## SPADL actions (`socceraction`)

In [ ]:
events = []
for i, match_id in enumerate(match_ids, 1):
    event = api.events(match_id)
    home_id = fixtures[fixtures["game_id"] == match_id]["home_team_id"].item()

    df_actions = spadl.statsbomb.convert_to_actions(event, home_team_id=home_id)
    df_actions = (
        spadl.add_names(df_actions)  # add actiontype and result names
        .merge(api.teams(game_id=match_id))  # add team names
        .merge(api.players(game_id=match_id))  # add player names
    )

    events.append(df_actions)
    print(f"  spadl actions {i}/{len(match_ids)}  match {match_id}  ({len(df_actions)} rows)")

event_df = pd.concat(events, ignore_index=True)
event_df.to_csv("data/event_df.csv")
print(f"data/event_df.csv  {event_df.shape}")